In [ ]:
import cupy as cp
from cupyx.scipy.signal import find_peaks
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from types import SimpleNamespace
from getMetaData import getMetaData
from getAScanBlockPreprocessed import getAScanBlockPreprocessed
from performMatchedFiltering import performMatchedFiltering
from determineOptimalPulse import determineOptimalPulse
from hilbert import hilbert
from reconstructSAFTimage import reconstructSAFTimage
import time

path = Path("<INPUT PATH HERE>")


SyntaxError: invalid syntax (2416355466.py, line 3)

Declaring positions

In [2]:
mp = 1
sl = cp.arange(1, 49)
sn = cp.arange(1, 19, 6)
rl = cp.arange(1, 49)
rn = cp.arange(1, 19, 3)

Loading metadata

In [ ]:
params, recoParams, reflectParams, expInfo, temp, ce, transformationMatrices, motorPosAvailable, geom, preComputes = getMetaData(path)

Reading, dividing and processing data

In [ ]:
applyFilter = False
transReco = 0
AScanBlockPreprocessed, mpBlock, slBlock, snBlock, rlBlock, rnBlock, senderPositionBlock, receiverPositionBlock, gainBlock, senderNormalBlock, receiverNormalBlock = getAScanBlockPreprocessed(params.rootMeasUniqueID, path, mp, sl, sn, rl, rn, geom, expInfo, reflectParams, applyFilter, transReco)

Matched filtering

In [ ]:
if AScanBlockPreprocessed.shape[1] > 0:
  AScanBlockMatchedFiltered = performMatchedFiltering(AScanBlockPreprocessed, rl, rn, rlBlock, rnBlock, geom, expInfo, preComputes)
else:
  warnings.warn("A-Scan Block is empty!")

stage1 = AScanBlockMatchedFiltered

Reconstructing an image using SAFT

Declaring parameters

In [ ]:
recoParams.startPoint = cp.array([[-0.17, -0.17, -0.03]])
recoParams.endPoint = cp.array([[0.17, 0.17, -0.03]])
recoParams.resolution = 0.001

recoParams.soundSpeed = temp.expectedSOSWater
recoParams.sampleRate = expInfo.SampleRate

recoParams.maxAngle = 90
recoParams.minAngle = 5
recoParams.sosWindowTransmission = cp.array([[1450, 1550]])

recoParams.snrThreshold = 3

Data restriction by angle filtering

In [ ]:
inbetweenAngle = cp.degrees(cp.arccos(cp.sum(senderNormalBlock * receiverNormalBlock, axis=0)))
AScanBlockMatchedFiltered = AScanBlockMatchedFiltered[:, (inbetweenAngle < recoParams.maxAngle) & (inbetweenAngle > recoParams.minAngle)]
senderPositionBlock = senderPositionBlock[:, (inbetweenAngle < recoParams.maxAngle) & (inbetweenAngle > recoParams.minAngle)]
receiverPositionBlock = receiverPositionBlock[:, (inbetweenAngle < recoParams.maxAngle) & (inbetweenAngle > recoParams.minAngle)]

Simplified data preprocessing

0. Remove DC trend + initial check for defect a-scans

In [ ]:
AScanBlockMatchedFiltered = AScanBlockMatchedFiltered - cp.mean(AScanBlockMatchedFiltered, 0)

maxVal = cp.max(cp.abs(AScanBlockMatchedFiltered), axis=0)
stdVal = cp.std(AScanBlockMatchedFiltered, axis=0, ddof=1)
meanVal = cp.mean(AScanBlockMatchedFiltered)
ascanMapValue = meanVal * stdVal
valid = cp.ones(AScanBlockMatchedFiltered.shape[1], dtype=cp.bool_)
valid[(stdVal == 0) & (maxVal != 0)] = False
valid[(maxVal / stdVal) < recoParams.snrThreshold] = False
AScanBlockMatchedFiltered = AScanBlockMatchedFiltered[:, valid]
senderPositionBlock = senderPositionBlock[:, valid]
receiverPositionBlock = receiverPositionBlock[:, valid]

1. Cut out the transmission signal

In [ ]:
distDirect = cp.sqrt(cp.sum((senderPositionBlock - receiverPositionBlock) ** 2, axis=0))
transmissionWindowEnd = cp.round((distDirect / recoParams.sosWindowTransmission[0, 0]) * recoParams.sampleRate)
transmissionWindowEnd[transmissionWindowEnd > AScanBlockMatchedFiltered.shape[0]] = AScanBlockMatchedFiltered.shape[0]

for idx in range(AScanBlockMatchedFiltered.shape[1]):
    end_idx = int(transmissionWindowEnd[idx])
    AScanBlockMatchedFiltered[:end_idx, idx] = cp.random.randn(end_idx)

2. Take envelope + find peaks + take strongest peaks + normalize + convolve with optimal pulse

In [ ]:
sincPeak_ft, optPulseFactor = determineOptimalPulse(recoParams.resolution, 96 , 1 / recoParams.sampleRate, AScanBlockMatchedFiltered.shape[0])
AScanBlockMatchedFiltered = cp.abs(hilbert(AScanBlockMatchedFiltered))

stage2 = AScanBlockMatchedFiltered

for idx in range(AScanBlockMatchedFiltered.shape[1]):
    col = AScanBlockMatchedFiltered[:, idx]
    peakInds, properties = find_peaks(col)
    peakVals = col[peakInds]
    sortOrder = cp.argsort(-peakVals)
    peakInds = peakInds[sortOrder]
    newAscan = cp.zeros(col.shape)
    numPeaksToUse = min(100, peakInds.shape[0])
    newAscan[peakInds[:numPeaksToUse]] = 1
    newAscan = cp.fft.ifft(cp.fft.fft(newAscan) * sincPeak_ft)
    AScanBlockMatchedFiltered[:, idx] = newAscan

stage3 = AScanBlockMatchedFiltered

SAFT core functionality

In [ ]:
cp.cuda.Stream.null.synchronize()
start = time.perf_counter()
img = reconstructSAFTimage(AScanBlockMatchedFiltered, senderPositionBlock, receiverPositionBlock, recoParams)
cp.cuda.Stream.null.synchronize()
elapsed = time.perf_counter() - start

Display performance statistics

In [ ]:
print(f"Reconstruction duration: {elapsed:.2f} seconds")
print(f"Image size: {img.shape[0]} x {img.shape[1]} x {img.shape[2]}")
print(f"number of A-Scans: {AScanBlockMatchedFiltered.shape[1]}")
print(f"Throughput: {AScanBlockMatchedFiltered.shape[1] * img.shape[0] * img.shape[1] / (elapsed * (1000**3))}")

Visualize reconstructed volume

In [ ]:
plt.figure()
plt.imshow(cp.asnumpy(img[:, :, 0]), cmap='gray')  
plt.axis('image')
plt.show()

A-Scan processing stage comparison table

In [ ]:
stages = [AScanBlockPreprocessed, stage1, stage2, stage3]
stageNames = ['Raw (preprocessed)', 'Matched filtered', 'Envelope (Hilbert)', 'Peak-sinc (sparse)']
stageDesc = [
    'Output of getAScanBlockPreprocessed — DC removed, gain applied',
    'After matched filtering — broadband signal, full dynamic range',
    'Absolute envelope via Hilbert transform — positive, smooth',
    'Top-100 peaks convolved with sinc — sparse spike train'
]

nStages = len(stages)
rows = []

for i in range(nStages):
    d = stages[i].astype(cp.float64)
    nSamples, nAscans = d.shape
    memMB = nSamples * nAscans * 8 / (1024**2)
    absD = cp.abs(d)
    thresh = 0.01 * cp.max(absD)
    sparsity = 100 * cp.mean(absD < thresh)
    dynRange = 20 * cp.log10(cp.max(absD) / (cp.mean(absD[absD > 0]) + cp.finfo(cp.float64).eps))

    rows.append({
        'Stage': stageNames[i],
        'Dimensions': f'{nSamples} × {nAscans}',
        'Memory': f'{float(memMB):.1f} MB',
        'Sparsity_lt1pct': f'{float(sparsity):.1f}%',
        'DynamicRange': f'{float(dynRange):.1f} dB',
        'Description': stageDesc[i]
    })

T = pd.DataFrame(rows)

print('\n========== A-SCAN STAGE COMPARISON ==========\n')
print(T)